---

### 1. `RH`, `Rain`, `FFMC`, `DMC`, `DC`, `ISI`, `BUI`, `FWI` — Ye Independent Features hain ya Outputs?

* **`RH` aur `Rain` (Weather Features):**
* **`RH` (Relative Humidity):** Hawa me kitni nami (humidity) hai. Higher RH = fire ka chance kam.
* **`Rain`:** Kitni baarish hui hai (mm me).
* Ye dono fully **Independent Features** (Inputs) hain.


* **FWI System Components (`FFMC`, `DMC`, `DC`, `ISI`, `BUI`, `FWI`):**
* **`FFMC` (Fine Fuel Moisture Code):** Dry leaves/grass ki moisture content.
* **`DMC` (Duff Moisture Code) & `DC` (Drought Code):** Mitti aur gehri lakdiyon ki moisture.
* **`ISI` (Initial Spread Index):** Aag kitni tezi se failegi (wind speed + moisture par base).
* **`BUI` (Buildup Index):** Kitna fuel (sookhi lakdi/patti) jalne ke liye available hai.
* **`FWI` (Fire Weather Index):** Overall fire intensity indicator.



> **Independent vs Dependent:**
> * **Project Goal 1 (Regression Problem):** Agar aap prediction kar rahe ho ki **`FWI`** ki score value kya hogi, toh baki saare features (`Temperature`, `RH`, `Wind speed`, `Rain`, `FFMC`, etc.) **Independent Features** honge aur **`FWI`** aapka **Dependent Feature (Output)** banega.
> * **Project Goal 2 (Classification Problem):** Agar aap aag lagne ya na lagne ka prediction kar rahe ho, toh `Classes` target output hoga aur baki sab (`FWI` samet) inputs honge.
> 
> 

---

### 2. Classes ka Kya Matlab Hai?

Dataset description me likha hai: *"The 244 instances have been classified into fire (138 classes) and not fire (106 classes) classes"*.

* **Classes** yahan par aapka **Target Output (Classification Label)** hai.
* Iska matlab hai ki pure dataset me total **244 rows (instances)** hain:
* **`fire` (138 rows):** Jismey jungle me aag lagi thi.
* **`not fire` (106 rows):** Jismey aag nahi lagi thi.


* Jab sir Classification model banayenge, toh ye `Classes` column predictor model ka final output (**Target Variable $y$**) banega.

---

### Step 1: Data Cleaning Me Sir Sabse Pehle Kya Karenge?

Algerian Forest Fires dataset me kafi dirt/formatting issues hote hain, isliye Data Cleaning me main steps ye hote hain:

1. **Header Clean-up & Extra Rows Removal:** Dataset me do alag regions (`Bejaia` aur `Sidi Bel-abbes`) ka data hai, aur dono regions ke beech me second header row aati hai, jise drop karna padega.
2. **Space / Trimming Issues:** `Classes` ya column names ke aage/peeche extra spaces hote hain (jaise `'not fire '`), unhe `.strip()` karke normalise karna padega.
3. **Region Column Add Karna:** Pehle 122 rows Bejaia region ke hain aur agle 122 Sidi Bel-abbes ke. Sir `Region` naam ka ek naya feature banayenge (`0` for Bejaia, `1` for Sidi Bel-abbes).
4. **Data Types Fix Karna:** Saare columns initially string format me read hote hain, unhe numeric (`int` / `float`) me convert karna padega.
5. **Null / Missing Values:** Dataset me agar koi missing value hai toh handle karna.

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
# reading csv file

dataset = pd.read_csv('Algerian_forest_fires_dataset_UPDATE.csv', header=1)

In [3]:
dataset.head()

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes
0,01,06,2012,29,57,18,0,65.7,3.4,7.6,1.3,3.4,0.5,not fire
1,02,06,2012,29,61,13,1.3,64.4,4.1,7.6,1,3.9,0.4,not fire
2,03,06,2012,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire
3,04,06,2012,25,89,13,2.5,28.6,1.3,6.9,0,1.7,0,not fire
4,05,06,2012,27,77,16,0,64.8,3,14.2,1.2,3.9,0.5,not fire


## data cleaning process



In [4]:
dataset.isnull().sum()     # --> how many values are missing in which columns 

day            0
month          1
year           1
Temperature    1
 RH            1
 Ws            1
Rain           1
FFMC           1
DMC            1
DC             1
ISI            1
BUI            1
FWI            1
Classes        2
dtype: int64

In [5]:
dataset[dataset.isnull().any(axis=1)]

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes
122,Sidi-Bel Abbes Region Dataset,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
167,14,07,2012,37,37,18,0.2,88.9,12.9,14.6 9,12.5,10.4,fire,NaN


In [6]:
# the dataset is converted into two sets based on region

In [7]:
dataset.loc[:122, "Region"] = 0
dataset.loc[122:, "Region"] = 1

In [8]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   day          246 non-null    str    
 1   month        245 non-null    str    
 2   year         245 non-null    str    
 3   Temperature  245 non-null    str    
 4    RH          245 non-null    str    
 5    Ws          245 non-null    str    
 6   Rain         245 non-null    str    
 7   FFMC         245 non-null    str    
 8   DMC          245 non-null    str    
 9   DC           245 non-null    str    
 10  ISI          245 non-null    str    
 11  BUI          245 non-null    str    
 12  FWI          245 non-null    str    
 13  Classes      244 non-null    str    
 14  Region       246 non-null    float64
dtypes: float64(1), str(14)
memory usage: 29.0 KB


In [9]:
dataset[['Region']] = dataset[['Region']].astype(int)

In [10]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   day          246 non-null    str  
 1   month        245 non-null    str  
 2   year         245 non-null    str  
 3   Temperature  245 non-null    str  
 4    RH          245 non-null    str  
 5    Ws          245 non-null    str  
 6   Rain         245 non-null    str  
 7   FFMC         245 non-null    str  
 8   DMC          245 non-null    str  
 9   DC           245 non-null    str  
 10  ISI          245 non-null    str  
 11  BUI          245 non-null    str  
 12  FWI          245 non-null    str  
 13  Classes      244 non-null    str  
 14  Region       246 non-null    int64
dtypes: int64(1), str(14)
memory usage: 29.0 KB


In [11]:
## removing the null values
dataset = dataset.dropna().reset_index(drop=True)

In [12]:
dataset = dataset.drop(122).reset_index(drop=True)

In [13]:
dataset.iloc[[122]]

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes,Region
122,01,06,2012,32,71,12,0.7,57.1,2.5,8.2,0.6,2.8,0.2,not fire,1


In [14]:
## some columns name have the extra spacing so we gonna remove that by stripe

dataset.columns = dataset.columns.str.strip()
dataset.columns

Index(['day', 'month', 'year', 'Temperature', 'RH', 'Ws', 'Rain', 'FFMC',
       'DMC', 'DC', 'ISI', 'BUI', 'FWI', 'Classes', 'Region'],
      dtype='str')

In [15]:
## change the required columns as integer or float datatypes

dataset[['month', 'day', 'year', 'Temperature', 'RH', 'Ws']] = dataset[['month', 'day', 'year', 'Temperature', 'RH', 'Ws']].astype(int)

dataset.info()


<class 'pandas.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   day          243 non-null    int64
 1   month        243 non-null    int64
 2   year         243 non-null    int64
 3   Temperature  243 non-null    int64
 4   RH           243 non-null    int64
 5   Ws           243 non-null    int64
 6   Rain         243 non-null    str  
 7   FFMC         243 non-null    str  
 8   DMC          243 non-null    str  
 9   DC           243 non-null    str  
 10  ISI          243 non-null    str  
 11  BUI          243 non-null    str  
 12  FWI          243 non-null    str  
 13  Classes      243 non-null    str  
 14  Region       243 non-null    int64
dtypes: int64(7), str(8)
memory usage: 28.6 KB


In [18]:
float_cols = ['Rain', 'FFMC', 'DMC', 'DC', 'ISI', 'BUI', 'FWI']
for col in float_cols:
    dataset[col] = dataset[col].astype(str).str.strip().astype(float)


In [19]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   day          243 non-null    int64  
 1   month        243 non-null    int64  
 2   year         243 non-null    int64  
 3   Temperature  243 non-null    int64  
 4   RH           243 non-null    int64  
 5   Ws           243 non-null    int64  
 6   Rain         243 non-null    float64
 7   FFMC         243 non-null    float64
 8   DMC          243 non-null    float64
 9   DC           243 non-null    float64
 10  ISI          243 non-null    float64
 11  BUI          243 non-null    float64
 12  FWI          243 non-null    float64
 13  Classes      243 non-null    str    
 14  Region       243 non-null    int64  
dtypes: float64(7), int64(7), str(1)
memory usage: 28.6 KB


In [22]:
dataset['Classes'] = dataset['Classes'].str.strip()


In [23]:
dataset.to_csv('cleaned_forest_fire_dataset.csv', index=False)
